## Librerias

In [10]:
import pandas as pd
from scipy.stats import ttest_ind
from pathlib import Path


## Carga datos clean

In [11]:
df_ops = pd.read_csv('C:\\Users\\Usuario\\ProjecteData\\Equip_25\\Data\\clean_data 16-03-2026.csv', parse_dates=['insert_date'])

# inventario activo
df_active = df_ops[df_ops["has_availability"] == True].copy()

df_active.info()
df_active.head()

<class 'pandas.core.frame.DataFrame'>
Index: 7009 entries, 0 to 7536
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 7009 non-null   int64         
 1   name                         7006 non-null   object        
 2   description                  7009 non-null   object        
 3   host_id                      7009 non-null   int64         
 4   neighbourhood_name           7009 non-null   object        
 5   neighbourhood_district       7009 non-null   object        
 6   room_type                    7009 non-null   object        
 7   accommodates                 7009 non-null   int64         
 8   bathrooms                    6967 non-null   float64       
 9   bedrooms                     6972 non-null   float64       
 10  beds                         7003 non-null   float64       
 11  amenities_list               7009 non-null   obj

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date,reviews 80+
0,11964,A ROOM WITH A VIEW,Private bedroom in our attic apartment. Right ...,45553,Centro,(sin contestar),Private room,2,2.0,1.0,...,100.0,100.0,100.0,100.0,False,75.0,spain,Malaga,2018-07-31,True
1,21853,Bright and airy room,We have a quiet and sunny room with a good vie...,83531,C�rmenes,Latina,Private room,1,1.0,1.0,...,100.0,100.0,80.0,90.0,False,52.0,spain,Madrid,2020-01-10,True
2,32347,Explore Cultural Sights from a Family-Friendly...,Open French doors and step onto a plant-filled...,139939,San Vicente,Casco Antiguo,Entire home/apt,4,1.0,2.0,...,100.0,100.0,100.0,100.0,True,142.0,spain,Sevilla,2019-07-29,True
3,35379,Double 02 CasanovaRooms Barcelona,Room at a my apartment. Kitchen and 2 bathroom...,152232,l'Antiga Esquerra de l'Eixample,Eixample,Private room,2,2.0,1.0,...,100.0,100.0,100.0,90.0,True,306.0,spain,Barcelona,2020-01-10,True
4,35801,Can Torras Farmhouse Studio Suite,Lay in bed & watch sunlight change the mood of...,153805,Quart,(sin contestar),Private room,5,1.0,2.0,...,100.0,100.0,100.0,100.0,False,39.0,spain,Girona,2019-02-19,True


#### Definición de métricas

In [12]:
# ocupación
df_active["occ_30"] = (30 - df_active["availability_30"]) / 30
df_active["occ_60"] = (60 - df_active["availability_60"]) / 60
df_active["occ_90"] = (90 - df_active["availability_90"]) / 90
df_active["occ_365"] = (365 - df_active["availability_365"]) / 365

occupancy_cols = ["occ_30", "occ_60", "occ_90", "occ_365"]

# días ocupados 
df_active["occupied_days_30"] = 30 - df_active["availability_30"]   

monthly_occupancy_rate = round(
    (df_active["occupied_days_30"].sum() /
     (df_active.shape[0] * 30)) * 100,
    2
)

# ocupación promedio por horizonte temporal
occupancy_rates = (df_active[occupancy_cols].mean() * 100).round(2)   
df_active["is_instant_bookable"].value_counts()

is_instant_bookable
True     3996
False    3013
Name: count, dtype: int64

## Análisis / visualizaciones
Quin impacte té l'opció de reservar automàticament
(sense revisió del propietari) a la disponibilitat mitjana a cada ciutat?

#### Global
Ocupación media  (instant vs no)

In [13]:
global_comparison = (
    df_active
    .groupby("is_instant_bookable")["occ_30"]
    .agg(["mean", "median", "count"])
    .round(3)
)

global_comparison

,mean,median,count
is_instant_bookable,,,
False,0.579,0.667,3013
True,0.594,0.667,3996


In [14]:
global_comparison = (
    df_active
    .groupby("is_instant_bookable")["availability_30"]
    .agg(["mean", "median", "count"])
    .round(3)
)

global_comparison

,mean,median,count
is_instant_bookable,,,
False,12.629,10.0,3013
True,12.189,10.0,3996


A nivel global los alojamientos con instant booking presentan mayor ocupación media

#### Tamaño de muestra

In [15]:
city_room_comparison = (
    df_active
    .groupby(["city", "room_type", "is_instant_bookable"])["occ_30"]
    .mean()
    .unstack()
    .round(3)
)

# eliminar casos sin ambos grupos
city_room_comparison = city_room_comparison.dropna()

# calcular uplift
city_room_comparison["uplift"] = (
    (city_room_comparison[True] - city_room_comparison[False]) * 100
).round(1)

city_room_comparison = city_room_comparison.sort_values("uplift", ascending=False)
city_room_comparison

is_instant_bookable        False   True  uplift
city      room_type                            
Girona    Private room     0.367  0.556    18.9
          Entire home/apt  0.457  0.563    10.6
Malaga    Private room     0.544  0.645    10.1
Madrid    Shared room      0.467  0.539     7.2
Menorca   Entire home/apt  0.482  0.544     6.2
Barcelona Private room     0.602  0.654     5.2
Valencia  Entire home/apt  0.561  0.601     4.0
          Private room     0.528  0.559     3.1
Madrid    Private room     0.661  0.688     2.7
Mallorca  Private room     0.492  0.497     0.5
Barcelona Entire home/apt  0.617  0.619     0.2
Mallorca  Entire home/apt  0.580  0.564    -1.6
Sevilla   Private room     0.623  0.597    -2.6
Madrid    Entire home/apt  0.650  0.623    -2.7
Sevilla   Entire home/apt  0.531  0.504    -2.7
Malaga    Entire home/apt  0.601  0.573    -2.8
Menorca   Private room     0.744  0.538   -20.6
Barcelona Hotel room       0.800  0.577   -22.3
          Shared room      0.576  0.225   -35.1
Mallorca  Shared room      1.000  0.433   -56.7

“Instant booking no es una estrategia universalmente efectiva; su impacto depende del mercado.”
Impacto de instant booking positivo en Girona, moderado en Menorca y Valencia --> instant booking aumenta reservas / más relevante en mercados menos saturados

En Barcelona impacto casi nulo --> la demanda ya es alta, por lo que no necesita/afecta instant booking / otros factores dominan (ubicación, reviews, precios,etc)

En Madrid y Sevilla impacto negativo --> Sesgo de selección (listings de menor calidad usan instant booking) / mercado ya eficiente (fricción no es problema) / segmentación distinta (tipos de alojamientos diferentes)

#### Comparación por ciudad

In [16]:
sample_size = (
    df_active
    .groupby(["city", "is_instant_bookable"])["apartment_id"]
    .count()
    .unstack()
)

sample_size

is_instant_bookable,False,True
city,,
Barcelona,910,910
Girona,478,704
Madrid,670,868
Malaga,121,273
Mallorca,456,701
Menorca,75,86
Sevilla,128,272
Valencia,175,182


In [17]:
size_room = (
    df_active
    .groupby(["city", "room_type", "is_instant_bookable"])["apartment_id"]
    .count()
    .unstack()
)

# alinear índices
size_room = size_room.loc[city_room_comparison.index]

# filtrar muestras pequeñas
city_room_comparison_filtered = city_room_comparison[
    (size_room[True] >= 20) & (size_room[False] >= 20)
]
size_room

is_instant_bookable        False  True 
city      room_type                    
Girona    Private room      29.0   35.0
          Entire home/apt  449.0  663.0
Malaga    Private room      31.0   33.0
Madrid    Shared room        9.0    6.0
Menorca   Entire home/apt   72.0   78.0
Barcelona Private room     539.0  422.0
Valencia  Entire home/apt  128.0  141.0
          Private room      47.0   36.0
Madrid    Private room     303.0  230.0
Mallorca  Private room      41.0   52.0
Barcelona Entire home/apt  363.0  470.0
Mallorca  Entire home/apt  414.0  638.0
Sevilla   Private room      40.0   43.0
Madrid    Entire home/apt  358.0  623.0
Sevilla   Entire home/apt   87.0  220.0
Malaga    Entire home/apt   90.0  234.0
Menorca   Private room       3.0    8.0
Barcelona Hotel room         1.0   10.0
          Shared room        7.0    8.0
Mallorca  Shared room        1.0    3.0

#### Test estadístico

In [18]:
instant = df_active[df_active["is_instant_bookable"] == True]["occ_30"]
non_instant = df_active[df_active["is_instant_bookable"] == False]["occ_30"]

t_stat, p_value = ttest_ind(instant, non_instant, equal_var=False)

print("p-value:", p_value)

p-value: 0.12067335649403242


## Resultados / archivos generados

In [1]:
# DF BASE: SOLO INVENTARIO ACTIVO
# --------------------------------
df_canva = df_active.copy()

# MÉTRICA: OCUPACIÓN
# ----------------
df_canva["occ_30"] = (30 - df_canva["availability_30"]) / 30

# AGREGACIÓN PRINCIPAL
# --------------------------------
agg = (
    df_active
    .groupby(["city", "room_type", "is_instant_bookable"])
    .agg(
        availability_mean=("availability_30", "mean"),
        occupancy_mean=("occ_30", "mean"),
        listings=("apartment_id", "count")
    )
    .reset_index()
)

# PASAR A FORMATO COMPARABLE
# --------------------------------
pivot = agg.pivot_table(
    index=["city", "room_type"],
    columns="is_instant_bookable",
    values=["availability_mean", "occupancy_mean", "listings"]
)

# flatten columnas
pivot.columns = [
    f"{metric}_{str(col)}"
    for metric, col in pivot.columns
]

pivot = pivot.reset_index()


# LIMPIEZA (solo segmentos válidos)
# --------------------------------
pivot = pivot.dropna()

pivot = pivot[
    (pivot["listings_True"] >= 10) &
    (pivot["listings_False"] >= 10)
]

# CALCULAR UPLIFT
# --------------------------------
pivot["uplift_occ"] = (
    pivot["occupancy_mean_True"] - pivot["occupancy_mean_False"]
)

pivot["uplift_occ_%"] = (pivot["uplift_occ"] * 100).round(1)

# disponibilidad (opcional pero potente)
pivot["uplift_availability"] = (
    pivot["availability_mean_True"] - pivot["availability_mean_False"]
)

# FORMATO FINAL
# --------------------------------
pivot["occupancy_True_%"] = (pivot["occupancy_mean_True"] * 100).round(1)
pivot["occupancy_False_%"] = (pivot["occupancy_mean_False"] * 100).round(1)

pivot["availability_True"] = pivot["availability_mean_True"].round(1)
pivot["availability_False"] = pivot["availability_mean_False"].round(1)

final_df = pivot[[
    "city",
    "room_type",
    "availability_True",
    "availability_False",
    "occupancy_True_%",
    "occupancy_False_%",
    "uplift_occ",
    "uplift_occ_%",
    "listings_True",
    "listings_False"
]].sort_values("uplift_occ_%", ascending=False)

final_df

# EXPORTAR RESULTADOS
final_df.to_csv("C:\\Users\\Usuario\\Desktop\\PROYECTO\\operaciones_results.csv", index=False)

NameError: name 'df_active' is not defined